In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

print("Imports OK")

Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
metadata_standard=METADATA_STANDARDS['spatial_ecological']
orchestrator = Orchestrator(topology_name='fast')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

2026-02-20 20:19:10,673 - INFO - root - PlanExecutor initialized with topology: fast
2026-02-20 20:19:10,673 - INFO - root -   Players per step: 2
2026-02-20 20:19:10,674 - INFO - root -   Debate rounds: 1
2026-02-20 20:19:10,674 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-02-20 20:19:10,674 - INFO - root - Orchestrator initialized with topology: fast
2026-02-20 20:19:10,674 - INFO - root - ============================================================
2026-02-20 20:19:10,675 - INFO - root - GENERATING PLAN
2026-02-20 20:19:10,675 - INFO - root - Context: biota_multi
2026-02-20 20:19:10,675 - INFO - root - Context type: csv
2026-02-20 20:19:10,675 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-02-20 20:19:10,675 - INFO - root - Multi-resource: True
2026-02-20 20:19:10,676 - INFO - root - ============================================================
2026-02-20 20:19:10,676 - INFO - root - Auto-added 'metadata_generator' for final metadata generati

In [4]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [5]:
from pprint import pprint
pprint(plan_dicts)

[{'inputs': {},
  'outputs': ['context_overview'],
  'player': 'data_analyst',
  'rationale': 'Get an overview of the entire context including all resources.',
  'target_tables': [],
  'task': 'get_context_overview'},
 {'inputs': {},
  'outputs': ['context_schema'],
  'player': 'schema_expert',
  'rationale': 'Get the complete schema of the context including resources, '
               'fields, and relationships.',
  'target_tables': [],
  'task': 'get_context_schema'},
 {'inputs': {},
  'outputs': ['discovered_relationships'],
  'player': 'relationship_analyst',
  'rationale': 'Discover and validate relationships between resources.',
  'target_tables': [],
  'task': 'get_relationships'},
 {'inputs': {'context_overview': 'context_overview',
             'context_schema': 'context_schema',
             'discovered_relationships': 'discovered_relationships',
             'metadata_standard': 'metadata_standard'},
  'outputs': ['metadata_output'],
  'player': 'metadata_generator',
  'rati

In [ ]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
metadata_standard = METADATA_STANDARDS["spatial_ecological"]
executor = PlanExecutor(topology_name="fast")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name="spatial_ecological"
)

2026-02-20 20:22:49,120 - INFO - root - PlanExecutor initialized with topology: fast
2026-02-20 20:22:49,121 - INFO - root -   Players per step: 2
2026-02-20 20:22:49,121 - INFO - root -   Debate rounds: 1
2026-02-20 20:22:49,121 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-02-20 20:22:49,121 - INFO - root - ============================================================
2026-02-20 20:22:49,121 - INFO - root - STARTING PLAN EXECUTION
2026-02-20 20:22:49,122 - INFO - root - Context: biota_multi
2026-02-20 20:22:49,122 - INFO - root - Type: csv
2026-02-20 20:22:49,122 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-02-20 20:22:49,122 - INFO - root - Steps: 4
2026-02-20 20:22:49,122 - INFO - root - ============================================================
2026-02-20 20:22:49,123 - INFO - root - 
2026-02-20 20:22:49,123 - INFO - root - ==================== STEP 1/4 ====================
2026-02-20 20:22:49,123 - INFO - root - Task: get_context_overvi

In [7]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

('```json\n'
 '{\n'
 '    "title": "Biota Multi-Resource Dataset",\n'
 '    "description": "This dataset contains information about biota samples '
 'collected from various tidal flats and basins. It includes details about the '
 'abundance and above-ground fine dry matter of different species in each '
 'sample.",\n'
 '    "subject": "Biological Sampling Data",\n'
 '    "spatial_coverage": "Various tidal flats and basins in the '
 'Netherlands",\n'
 '    "spatial_resolution": "Geographic coordinates (x, y)",\n'
 '    "temporal_coverage": "2008-07-14 to 2023-09-25",\n'
 '    "temporal_resolution": "Daily sampling dates",\n'
 '    "methods": "Sampling methods include grid and random sampling. Data '
 'collection platforms are boat and walk.",\n'
 '    "format": "CSV",\n'
 '    "tables": {\n'
 '        "biota": {\n'
 '            "description": "Contains biota data including sample IDs, '
 'species IDs, abundance per square meter, and above-ground fine dry matter '
 'per square meter.",\